# ⚡🌷 LILY FASTWAN 5B STUDIO
Wan 2.2 **TI2V 5B + FastWan 3-Step**. Enable **T4 x2**, then run Cells **1 → 4**.


In [ ]:
from pathlib import Path
import os, subprocess, shutil

ROOT = Path('/kaggle/working/Wan2GP')
DATA = Path('/kaggle/temp/Wan2GP-data')
CKPTS = DATA / 'ckpts'
CACHE = DATA / 'cache'
LORAS = DATA / 'loras'
OUTPUTS = Path('/kaggle/working/Wan2GP-outputs')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Enable Kaggle GPU / T4 x2, let it restart, then rerun Cell 1.')

subprocess.run(['nvidia-smi'], check=True)
for p in (CKPTS, CACHE, LORAS, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

os.environ['WAN_CACHE_DIR'] = str(CACHE)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

if not (ROOT / '.git').exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/Wan2GP.git', str(ROOT)], check=True)

def attach(src, dst):
    if src.is_symlink():
        if src.resolve() == dst.resolve():
            return
        src.unlink()
    elif src.exists():
        for x in list(src.iterdir()):
            y = dst / x.name
            if not y.exists():
                shutil.move(str(x), str(y))
        shutil.rmtree(src)
    src.symlink_to(dst, target_is_directory=True)

attach(ROOT / 'ckpts', CKPTS)
attach(ROOT / 'loras', LORAS)
attach(ROOT / 'outputs', OUTPUTS)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', 'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], check=True)
print('✅ CELL 1 COMPLETE')


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

ROOT = Path('/kaggle/working/Wan2GP')
CACHE = Path('/kaggle/temp/Wan2GP-data/cache')
probe = "import json;import torch,torchvision,torchaudio;print(json.dumps({'torch':torch.__version__,'torchvision':torchvision.__version__,'torchaudio':torchaudio.__version__}))"
v = json.loads(subprocess.check_output([sys.executable, '-c', probe], text=True).strip().splitlines()[-1])
c = CACHE / 'constraints.txt'
c.write_text('\n'.join(f"{k}=={x.split('+')[0]}" for k, x in v.items()) + '\n')
env = os.environ.copy()
env['PIP_NO_CACHE_DIR'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', '-r', str(ROOT / 'requirements.txt'), '-c', str(c)], check=True, env=env)
subprocess.run([sys.executable, '-c', "import torch,numpy,mmgp,rembg,gradio;assert torch.cuda.is_available();print('GPU:',torch.cuda.get_device_name(0));print('✅ dependencies OK')"], check=True)
print('✅ CELL 2 COMPLETE')


In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request

ROOT = Path('/kaggle/working/Wan2GP')
P = Path('/kaggle/working/lily_fastwan5b_prewarm.py')
urllib.request.urlretrieve('https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_fastwan5b_prewarm.py', P)
print('📦 FastWan 5B helper updated.')
subprocess.run([sys.executable, '-u', str(P)], cwd=str(ROOT), env=os.environ.copy(), check=True)
print('✅ CELL 3 COMPLETE')


In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request

ROOT = Path('/kaggle/working/Wan2GP')
S = Path('/kaggle/working/lily_fastwan5b_studio.py')
urllib.request.urlretrieve('https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_fastwan5b_studio.py', S)
print('⚡ Studio updated. Wait for the gradio.live link…')
subprocess.run([sys.executable, '-u', str(S)], cwd=str(ROOT), env=os.environ.copy(), check=False)
